# Figure 1: prompt intersection on GPT-2

Two GPT-2 prompts, *My favorite physicist is* and *My favorite author is*, combined as a product.
200 completions are enumerated by best-first search (beam 30) over the locally normalised product and
scored under the local and the global ensemble; the right panels correlate both with the explicit
intersection prompt *My favorite physicist AND author is* (ρ = 0.090 local, 0.958 global over the 174
ensemble strings that neither single prompt alone produced).

Runs end to end on one GPU in a few minutes; set `GENERATE = False` to re-plot from `synthetic_example.json`.


In [ ]:
import json, re, random, shutil
from pathlib import Path

import numpy as np
import torch
from arsenal.datastructures import LocatorMaxHeap
from arsenal.maths import logsumexp

from genlm.control import PromptedLLM
from genlm.control.constant import EndOfSequence

# ---- configuration -------------------------------------------------------------------
MODEL        = "gpt2"
PROMPT_1     = "My favorite physicist is"
PROMPT_2     = "My favorite author is"
PROMPT_AND   = "My favorite physicist AND author is"   # explicit intersection (paper, Sec. 2); the figure used the capitalised conjunction
PROMPT_OR    = "My favorite physicist OR author is"    # explicit union (not plotted)
OP           = "prod"       # f-ensemble used for the search: min | prod | sum | max
T_ENSEMBLE   = 200          # completions to enumerate from the ensemble
T_SINGLE     = 20           # completions to enumerate from each prompt alone
BRANCHING_ENSEMBLE = 30     # beam width (the figure: 30; 174 ensemble strings survive the physicist/author filter)
BRANCHING_SINGLE   = 20
CORR_TOP_K   = None         # right panels: correlate over all ensemble strings (an int keeps the top-k by p(. | AND))
MAX_TOKENS   = 100
SEED         = 43
GENERATE     = True
DATA_FILE    = Path("synthetic_example.json")
FIG_FILE     = Path("synthetic_example.pdf")

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

set_seed(SEED)

## Enumeration

In [ ]:
def make_llm(prompt):
    llm = PromptedLLM.from_name(MODEL, backend="hf", temperature=1.0)
    llm.set_prompt_from_str(prompt)
    return llm

def to_np(w):
    return w.detach().cpu().numpy() if torch.is_tensor(w) else np.asarray(w)

def to_text(tokens):
    """Byte tokens -> str (EOS objects dropped)."""
    return b"".join(t for t in tokens if isinstance(t, bytes)).decode("utf-8", errors="ignore")

def is_complete(text):
    """A completion is done when it ends in a period that is not a title abbreviation."""
    return text.endswith(".") and not re.match(r".*\s(Prof|Dr|Mr|Mrs|[A-Z])\.", text)

# f in log space, applied element-wise to the models' (normalised) log-probabilities.
# These are the unweighted operators the figure was produced with; the paper's weighted
# generalised means (genlm.control.Ensemble) differ by a temperature for prod/sum.
F = {
    "min":  lambda x, y: np.minimum(x, y),
    "max":  lambda x, y: np.maximum(x, y),
    "prod": lambda x, y: x + y,
    "sum":  lambda x, y: np.logaddexp(x, y),
}

class Node:
    __slots__ = ("context", "logps", "score")
    def __init__(self, context, logps, score):
        self.context, self.logps, self.score = context, logps, score   # score: local (proposal) log-prob
    def __hash__(self):           return hash(self.context)
    def __eq__(self, other):      return self.context == other.context

async def best_first(llms, f, T, branching, max_tokens=MAX_TOKENS):
    """Yield the first `T` complete nodes in best-first order of the locally-normalised f-ensemble.

    `llms` share a tokenizer, so their next-token distributions align index-by-index.
    For a single model pass `f=None`.
    """
    agenda, seen, done = LocatorMaxHeap(), set(), 0
    root = Node((), (0.0,) * len(llms), 0.0)
    agenda[root] = root.score
    while agenda and done < T:
        node, _ = agenda.pop()
        if node.context in seen:
            continue
        seen.add(node.context)
        if is_complete(to_text(node.context)):
            done += 1
            yield node
            continue
        if len(node.context) >= max_tokens:
            continue
        lws = [await llm.logw_next(list(node.context)) for llm in llms]
        logps = [to_np(lw.normalize().weights) for lw in lws]
        step = logps[0] if f is None else f(*logps)
        step = step - logsumexp(step)                      # local normalisation -> proposal
        decode = lws[0].decode
        for idx in np.argpartition(-step, min(branching, len(step) - 1))[:branching]:
            tok = decode[idx]
            if isinstance(tok, EndOfSequence):             # EOS ends the string; we stop on "." instead
                continue
            agenda[Node(node.context + (tok,),
                        tuple(lp + l[idx] for lp, l in zip(node.logps, logps)),
                        node.score + float(step[idx]))] = node.score + float(step[idx])

def chart(nodes, key):
    """Accumulate exp(key(node)) by generated text (different tokenisations of one string add up)."""
    out = {}
    for n in nodes:
        out[to_text(n.context)] = out.get(to_text(n.context), 0.0) + float(np.exp(key(n)))
    z = sum(out.values())
    return {k: v / z for k, v in out.items()}

## Generate and score

In [ ]:
async def generate_and_score():
    llm1, llm2 = make_llm(PROMPT_1), make_llm(PROMPT_2)
    llm_and, llm_or = make_llm(PROMPT_AND), make_llm(PROMPT_OR)
    f = F[OP]

    physicists = chart([n async for n in best_first([llm1], None, T_SINGLE, BRANCHING_SINGLE)], key=lambda n: n.score)
    authors    = chart([n async for n in best_first([llm2], None, T_SINGLE, BRANCHING_SINGLE)], key=lambda n: n.score)
    ens_nodes  = [n async for n in best_first([llm1, llm2], f, T_ENSEMBLE, BRANCHING_ENSEMBLE)]
    local      = chart(ens_nodes, key=lambda n: n.score)                  # proposal: product of local normalisers
    global_    = chart(ens_nodes, key=lambda n: f(*n.logps))             # f over whole-string log-probs

    records = []
    for text in dict.fromkeys(list(local) + list(physicists) + list(authors)):   # ordered, unique
        toks = llm1.tokenize(text)
        records.append(dict(
            text=text,
            logp1=float(await llm1.complete(toks)),
            logp2=float(await llm2.complete(toks)),
            logp_and=float(await llm_and.complete(toks)),
            logp_or=float(await llm_or.complete(toks)),
            local_prob=local.get(text, 0.0),
            global_prob=global_.get(text, 0.0),
            in_ensemble=text in local,
            in_physicists=text in physicists,
            in_authors=text in authors,
        ))
    return dict(config=dict(model=MODEL, prompts=[PROMPT_1, PROMPT_2, PROMPT_AND, PROMPT_OR], op=OP,
                            T_ensemble=T_ENSEMBLE, T_single=T_SINGLE, branching_ensemble=BRANCHING_ENSEMBLE, branching_single=BRANCHING_SINGLE,
                            max_tokens=MAX_TOKENS, seed=SEED),
                records=records)

if GENERATE:
    data = await generate_and_score()
    DATA_FILE.write_text(json.dumps(data, indent=1))
    print(f"wrote {len(data['records'])} strings -> {DATA_FILE}")
data = json.loads(DATA_FILE.read_text())
R = data["records"]
print(f"{sum(r['in_ensemble'] for r in R)} ensemble completions, "
      f"{sum(r['in_physicists'] for r in R)} physicist-only prompt completions, "
      f"{sum(r['in_authors'] for r in R)} author-only prompt completions")
for name, key in (("global", "global_prob"), ("local", "local_prob")):
    top = sorted((r for r in R if r["in_ensemble"]), key=lambda r: -r[key])[:10]
    print(f"\ntop-10 by {name}:"); [print(f"  {r[key]:.4f}  {r['text']!r}") for r in top]

## Figure

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from scipy import stats
from scipy.stats import pearsonr, linregress

plt.rcParams.update({"font.size": 18, "axes.labelsize": 18, "xtick.labelsize": 16, "ytick.labelsize": 16,
                     "font.family": "serif", "text.usetex": shutil.which("latex") is not None})
ORANGE_TEAL = LinearSegmentedColormap.from_list("OrangeTeal", ["#d95f02", "#ffffff", "#1b9e77"])

text   = np.array([r["text"] for r in R])
logp1  = np.array([r["logp1"] for r in R]);        logp2 = np.array([r["logp2"] for r in R])
p_loc  = np.array([r["local_prob"] for r in R]);   p_glo = np.array([r["global_prob"] for r in R])
p_and  = np.exp([r["logp_and"] for r in R])
in_ens = np.array([r["in_ensemble"] for r in R])
only_p = np.array([r["in_physicists"] and not r["in_authors"] and not r["in_ensemble"] for r in R])
only_a = np.array([r["in_authors"] and not r["in_physicists"] and not r["in_ensemble"] for r in R])
globalness = np.clip(np.where(p_loc > 0, p_glo / np.where(p_loc > 0, p_loc, 1), 0.5), 0, 1)

# left panel shows the 75 most probable strings under either scoring, plus the single-prompt completions
top = set(np.argsort(-p_glo)[:75]) | set(np.argsort(-p_loc)[:75])
show_ens = np.array([i in top and in_ens[i] for i in range(len(R))])

fig = plt.figure(figsize=(20, 9))
gs = fig.add_gridspec(2, 2, width_ratios=[1.2, 0.5])
ax = fig.add_subplot(gs[:, 0])
style = dict(cmap=ORANGE_TEAL, vmin=0, vmax=1, alpha=0.85, s=250, edgecolors="white", linewidths=0.8, zorder=5)
sc = ax.scatter(logp1[show_ens], logp2[show_ens], c=globalness[show_ens], marker="o", **style)
ax.scatter(logp1[only_p], logp2[only_p], color="0.25", marker="s", **{k: v for k, v in style.items() if k not in ("cmap", "vmin", "vmax")})
ax.scatter(logp1[only_a], logp2[only_a], color="0.25", marker="^", **{k: v for k, v in style.items() if k not in ("cmap", "vmin", "vmax")})
cb = fig.colorbar(sc, ax=ax, location="left", shrink=0.6, pad=0.08)
cb.set_label(r"$p_{\mathrm{ens,global}}/p_{\mathrm{ens,local}}$", fontsize=22, labelpad=16)
ax.legend(handles=[
    Line2D([0], [0], marker="s", color="w", markerfacecolor="0.25", markeredgecolor="white", markersize=10, label=f'"{PROMPT_1}..."'),
    Line2D([0], [0], marker="^", color="w", markerfacecolor="0.25", markeredgecolor="white", markersize=10, label=f'"{PROMPT_2}..."'),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="0.5",  markeredgecolor="white", markersize=10, label=f"Ensemble: {OP}(physicists, authors)"),
], loc="upper left", fontsize=20)
ax.set_xlim(-30, 0); ax.set_ylim(-30, 0); ax.grid(True, alpha=0.3); ax.set_axisbelow(True)
ax.set_xlabel(r"$\log p(\cdot \mid$ " + PROMPT_1 + ")", fontsize=22)
ax.set_ylabel(r"$\log p(\cdot \mid$ " + PROMPT_2 + ")", fontsize=22)

# annotations: authors in a left column, everything else in a right column, slots ordered by the point's y
def annotate(names, x_text, ha, y_top=-2.5, y_bottom=-27.5):
    idx = [int(np.where(text == n)[0][0]) for n in names if n in text and (show_ens | only_p | only_a)[np.where(text == n)[0][0]]]
    idx.sort(key=lambda i: -logp2[i])
    slots = np.linspace(y_top, y_bottom, max(len(idx), 2))[: len(idx)]
    for i, y in zip(idx, slots):
        ax.annotate(text[i], xy=(logp1[i], logp2[i]), xytext=(x_text, y), fontsize=20, ha=ha, va="center",
                    bbox=dict(boxstyle="round,pad=0.4", facecolor=ORANGE_TEAL(globalness[i]) if in_ens[i] else "0.85", alpha=0.3, edgecolor="black"),
                    arrowprops=dict(arrowstyle="->", lw=1.2, color="black", alpha=0.6))
annotate([" James Baldwin.", " Robert Jordan."], x_text=-29, ha="left", y_top=-9, y_bottom=-21)
annotate([" Carl Sagan.", " David Hilbert.", " Einstein.", " Richard Feynman.", " Bill Nye.", " a great guy.", " a very good one."], x_text=-1, ha="right")

# right panels: ensemble strings vs. the explicit intersection prompt, without the strings that
# exactly one of the single prompts also enumerates (the figure's selection; 174 strings)
in_p = np.array([r["in_physicists"] for r in R]); in_a = np.array([r["in_authors"] for r in R])
ok = in_ens & (p_loc > 0) & (p_glo > 0) & ~(in_p ^ in_a)
sel = np.array(sorted(np.where(ok)[0], key=lambda i: -p_and[i])[:CORR_TOP_K] if CORR_TOP_K else np.where(ok)[0])
y = np.log(p_and[sel])
def panel(axp, x, xlabel, pos):
    axp.scatter(x, y, c=globalness[sel], cmap=ORANGE_TEAL, vmin=0, vmax=1, s=50, edgecolors="white", linewidths=0.8, zorder=5)
    slope, icpt, *_ = linregress(x, y); xl = np.linspace(x.min(), x.max(), 100); yl = slope * xl + icpt
    res = y - (slope * x + icpt); se = np.sqrt((res**2).sum() / (len(x) - 2)) * np.sqrt(1/len(x) + (xl - x.mean())**2 / ((x - x.mean())**2).sum())
    axp.fill_between(xl, yl - stats.t.ppf(0.975, len(x)-2)*se, yl + stats.t.ppf(0.975, len(x)-2)*se, color="grey", alpha=0.2)
    axp.plot(xl, yl, color="grey", lw=2.5)
    rho, p = pearsonr(x, y); stars = "***" if p < 1e-3 else "**" if p < 1e-2 else "*" if p < 0.05 else ""
    axp.text(*pos, rf"$\rho = {rho:.3f}${stars}", transform=axp.transAxes, fontsize=22, ha="right",
             va="bottom" if pos[1] < 0.5 else "top", bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))
    axp.set_xlabel(xlabel, fontsize=22); axp.set_ylabel(r"$\log p(\cdot \mid$ My favorite physicist" + "\n" + r"$\mathbf{and}$ author is)", fontsize=16, labelpad=12)
    axp.grid(True, alpha=0.3); axp.set_axisbelow(True)
    return rho, p
rho_loc, p_loc_ = panel(fig.add_subplot(gs[0, 1]), np.log(p_loc[sel]), r"$\log p_{\mathrm{ens,local}}$",  (0.96, 0.05))
rho_glo, p_glo_ = panel(fig.add_subplot(gs[1, 1]), np.log(p_glo[sel]), r"$\log p_{\mathrm{ens,global}}$", (0.96, 0.05))

fig.tight_layout(); fig.savefig(FIG_FILE, dpi=300, bbox_inches="tight"); plt.show()
print(f"rho(log p_local,  log p_AND) = {rho_loc:.3f} (p={p_loc_:.2e})")
print(f"rho(log p_global, log p_AND) = {rho_glo:.3f} (p={p_glo_:.2e})")
print(f"saved {FIG_FILE}")